In [ ]:
code = 'LONG_BUTTERFLY_SPREAD'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def LONG_BUTTERFLY_SPREAD(bt, start_time, end_time, option_type, body_om, wing_width, sl, target, dte1re, dte2re, dte3re, dte4re, dte5re):
    try:
        option_type = str(option_type).upper().strip()
        if option_type not in ('CE', 'PE'): return None

        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)

        def run_cycle(from_dt):

            body_scrip, _, future_price, entry_dt = bt.get_strike(from_dt, end_dt, om=body_om, only=option_type)
            if body_scrip is None: return None

            step = int(wing_width) * bt.gap
            if step <= 0: return None

            lower_scrip = f"{get_strike(body_scrip) - step}{option_type}"
            upper_scrip = f"{get_strike(body_scrip) + step}{option_type}"

            lower_data = bt.get_single_leg_data(entry_dt, end_dt, lower_scrip)
            body_data = bt.get_single_leg_data(entry_dt, end_dt, body_scrip)
            upper_data = bt.get_single_leg_data(entry_dt, end_dt, upper_scrip)

            common_dt = np.intersect1d(lower_data['date_time'].values, body_data['date_time'].values)
            common_dt = np.intersect1d(common_dt, upper_data['date_time'].values)

            if len(common_dt) == 0:
                return None

            lower_data = lower_data[np.isin(lower_data['date_time'].values, common_dt)]
            body_data = body_data[np.isin(body_data['date_time'].values, common_dt)]
            upper_data = upper_data[np.isin(upper_data['date_time'].values, common_dt)]

            cycle_entry_time = lower_data['date_time'].iloc[0]

            lower_price = lower_data['close'].iloc[0]
            body_price = body_data['close'].iloc[0]
            upper_price = upper_data['close'].iloc[0]

            ### long butterfly is a debit structure - buy 1 lower, sell 2 body, buy 1 upper
            debit = (lower_price + upper_price) - (2 * body_price)
            if debit <= 0:
                return None

            close_value_list = ((lower_data['close'].values + upper_data['close'].values) - (2 * body_data['close'].values)).tolist()
            eod_value = close_value_list[-1]

            slipage = bt.Cal_slipage(lower_price + upper_price + (2 * body_price))
            no_exit_pnl = round((eod_value - debit) - slipage, 2)

            ### debit structure - sl is a fall below cost, target is a rise above it (0 = disabled)
            sl_price = debit * (1 - (sl/100)) if sl else None
            target_price = debit * (1 + (target/100)) if target else None

            exit_reason, exit_dt, exit_pnl = 'EOD', '', no_exit_pnl

            for i, ele in enumerate(close_value_list if (sl_price is not None) or (target_price is not None) else []):
                if (sl_price is not None) and (ele <= sl_price):
                    exit_reason, exit_dt = 'SL', lower_data['date_time'].iloc[i]
                    exit_pnl = round((ele - debit) - slipage, 2)
                    break
                if (target_price is not None) and (ele >= target_price):
                    exit_reason, exit_dt = 'TARGET', lower_data['date_time'].iloc[i]
                    exit_pnl = round((ele - debit) - slipage, 2)
                    break

            legs = f"({lower_scrip}, 2x{body_scrip}, {upper_scrip})"
            return [cycle_entry_time, legs, debit, exit_reason, exit_dt, exit_pnl], future_price

        cycle = run_cycle(start_dt)
        if cycle is None: return None
        trade, future_price = cycle

        entry_time = trade[0]
        trades = [trade]
        exit_time = trade[4] if trade[3] in ('SL', 'TARGET') else ''

        re_entries_left = {1: dte1re, 2: dte2re, 3: dte3re, 4: dte4re, 5: dte5re}
        blank_slot = ['', '', '', '', '', 0]

        re_trades = []
        for re_no in range(max_re):

            if exit_time and (exit_time < end_dt - datetime.timedelta(minutes=5)):

                rdte = int(dte_file.loc[pd.to_datetime(exit_time.date()), bt.index])

                ### budget spent for this dte - park the re-entry near the close so it carries into the next dte
                if re_entries_left.get(rdte, 0) == 0:
                    if rdte == 1:
                        exit_time = ''
                        re_trades.extend(blank_slot)
                        continue

                    exit_time = max(exit_time, (datetime.datetime.combine(exit_time.date(), bt.meta_end_time) - datetime.timedelta(minutes=15)))
                else:
                    re_entries_left[rdte] -= 1

                cycle = run_cycle(exit_time)
                if cycle is None:
                    exit_time = ''
                    re_trades.extend(blank_slot)
                    continue

                trade, _ = cycle
                trades.append(trade)
                re_trades.extend(trade)
                exit_time = trade[4] if trade[3] in ('SL', 'TARGET') else ''
            else:
                re_trades.extend(blank_slot)

        total_pnl = round(sum(t[-1] for t in trades), 2)
        re_count = len(trades) - 1

        dte_pnl = {5: 0, 4: 0, 3: 0, 2: 0, 1: 0}
        for t in trades:
            tdte = int(dte_file.loc[pd.to_datetime(t[0].date()), bt.index])
            if tdte in dte_pnl: dte_pnl[tdte] += t[-1]
        dte_pnl_list = [round(dte_pnl[d], 2) for d in (5, 4, 3, 2, 1)]

        return [code, bt.index, start_time, end_time, option_type, body_om, wing_width, sl, target, dte1re, dte2re, dte3re, dte4re, dte5re, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time, future_price] + trades[0] + re_trades + [total_pnl, re_count] + dte_pnl_list

    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, option_type, body_om, wing_width, sl, target])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)
            dte_file = get_dte_file(pickle_path)
            max_re = 10

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_OptionType/P_BodyOM/P_WingWidth/P_SL/P_Target/P_Dte1Re/P_Dte2Re/P_Dte3Re/P_Dte4Re/P_Dte5Re/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future'

            for r in range(max_re+1):
                log_cols += f'/T{r}.Time/T{r}.Legs/T{r}.Debit/T{r}.Exit.Reason/T{r}.Exit.Time/T{r}.PNL'
            log_cols += '/Total.PNL/Re.Count/Dte5.PNL/Dte4.PNL/Dte3.PNL/Dte2.PNL/Dte1.PNL'
            log_cols = log_cols.split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [LONG_BUTTERFLY_SPREAD(wbt, row['entry_time'], row['exit_time'], row['option_type'], row['body_om'], row['wing_width'], row['sl'], row['target'], row['dte1re'], row['dte2re'], row['dte3re'], row['dte4re'], row['dte5re']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del wbt
                    gc.collect()

                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))